In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SQLContext 

In [2]:
spark = SparkSession \
    .builder \
    .master("local[*]") \
    .appName("Exercicio Intro") \
    .getOrCreate()

df_nomes = spark.read.csv("nomes_aleatorios.txt")
df_nomes.show(5)

+------------------+
|               _c0|
+------------------+
|       Roy Fuhrman|
|Roosevelt Kirchner|
|   Milton Longoria|
|     Thomas Ringel|
|     John Mckinney|
+------------------+
only showing top 5 rows


In [3]:
# Renomeando a coluna _c0 para Nomes
df_nomes = df_nomes.withColumnRenamed("_c0", "Nomes")

# Exibindo o schema
df_nomes.printSchema()

# Mostrando as 10 primeiras linhas
df_nomes.show(10)

root
 |-- Nomes: string (nullable = true)

+-------------------+
|              Nomes|
+-------------------+
|        Roy Fuhrman|
| Roosevelt Kirchner|
|    Milton Longoria|
|      Thomas Ringel|
|      John Mckinney|
|Christopher Timothy|
|       Lisa Hammond|
|       Olga Naranjo|
|     Thomas Padgett|
|      Samuel Silver|
+-------------------+
only showing top 10 rows


In [4]:
from pyspark.sql import functions as F

df_nomes = df_nomes.withColumn(
    "Escolaridade",
    F.when(F.rand() < 1/3, "Fundamental")
     .when(F.rand() < 2/3, "Médio")
     .otherwise("Superior")
)

df_nomes.show(10)

+-------------------+------------+
|              Nomes|Escolaridade|
+-------------------+------------+
|        Roy Fuhrman|       Médio|
| Roosevelt Kirchner| Fundamental|
|    Milton Longoria| Fundamental|
|      Thomas Ringel| Fundamental|
|      John Mckinney|    Superior|
|Christopher Timothy|       Médio|
|       Lisa Hammond|       Médio|
|       Olga Naranjo| Fundamental|
|     Thomas Padgett|    Superior|
|      Samuel Silver| Fundamental|
+-------------------+------------+
only showing top 10 rows


In [5]:
df_nomes = df_nomes.withColumn(
    "Pais",
    F.when(F.rand() < 1/13, "Brasil")
     .when(F.rand() < 2/13, "Argentina")
     .when(F.rand() < 3/13, "Chile")
     .when(F.rand() < 4/13, "Uruguai")
     .when(F.rand() < 5/13, "Paraguai")
     .when(F.rand() < 6/13, "Bolívia")
     .when(F.rand() < 7/13, "Peru")
     .when(F.rand() < 8/13, "Equador")
     .when(F.rand() < 9/13, "Colômbia")
     .when(F.rand() < 10/13, "Venezuela")
     .when(F.rand() < 11/13, "Guiana")
     .when(F.rand() < 12/13, "Suriname")
     .otherwise("Guiana Francesa")
)

df_nomes.show(10)

+-------------------+------------+---------+
|              Nomes|Escolaridade|     Pais|
+-------------------+------------+---------+
|        Roy Fuhrman|       Médio| Paraguai|
| Roosevelt Kirchner| Fundamental|Argentina|
|    Milton Longoria| Fundamental|  Uruguai|
|      Thomas Ringel| Fundamental|  Uruguai|
|      John Mckinney|    Superior|    Chile|
|Christopher Timothy|       Médio|    Chile|
|       Lisa Hammond|       Médio|    Chile|
|       Olga Naranjo| Fundamental| Paraguai|
|     Thomas Padgett|    Superior|    Chile|
|      Samuel Silver| Fundamental|Argentina|
+-------------------+------------+---------+
only showing top 10 rows


In [6]:
df_nomes = df_nomes.withColumn(
    "AnoNascimento",
    (F.floor(F.rand() * (2010 - 1945 + 1)) + 1945).cast("int")
)

df_nomes.show(10)

+-------------------+------------+---------+-------------+
|              Nomes|Escolaridade|     Pais|AnoNascimento|
+-------------------+------------+---------+-------------+
|        Roy Fuhrman|       Médio| Paraguai|         1952|
| Roosevelt Kirchner| Fundamental|Argentina|         1947|
|    Milton Longoria| Fundamental|  Uruguai|         2009|
|      Thomas Ringel| Fundamental|  Uruguai|         1996|
|      John Mckinney|    Superior|    Chile|         1947|
|Christopher Timothy|       Médio|    Chile|         1974|
|       Lisa Hammond|       Médio|    Chile|         1997|
|       Olga Naranjo| Fundamental| Paraguai|         1967|
|     Thomas Padgett|    Superior|    Chile|         1992|
|      Samuel Silver| Fundamental|Argentina|         1979|
+-------------------+------------+---------+-------------+
only showing top 10 rows


In [7]:
df_select = df_nomes.select("*").where(F.col("AnoNascimento") >= 2000)
df_select.show(10)

+------------------+------------+---------+-------------+
|             Nomes|Escolaridade|     Pais|AnoNascimento|
+------------------+------------+---------+-------------+
|   Milton Longoria| Fundamental|  Uruguai|         2009|
|     Margaret King|       Médio|    Chile|         2005|
|  Matthew Trembley|       Médio|  Uruguai|         2001|
|         Mary Cole|       Médio|  Uruguai|         2010|
|     Marita Hively|       Médio|  Bolívia|         2009|
|   Carissa Stanley|       Médio|    Chile|         2004|
|   Richard Bundage| Fundamental|  Uruguai|         2010|
|      Michael Dunn| Fundamental|  Uruguai|         2003|
|Marybeth Eberheart|       Médio|  Uruguai|         2007|
|   Deborah Lizotte|       Médio|Argentina|         2004|
+------------------+------------+---------+-------------+
only showing top 10 rows


In [8]:
# Criando uma view temporária
df_nomes.createOrReplaceTempView("pessoas")

# Usando SQL para selecionar quem nasceu neste século
df_select_sql = spark.sql("""
    SELECT * FROM pessoas
    WHERE AnoNascimento >= 2000
""")

df_select_sql.show(10)

+------------------+------------+---------+-------------+
|             Nomes|Escolaridade|     Pais|AnoNascimento|
+------------------+------------+---------+-------------+
|   Milton Longoria| Fundamental|  Uruguai|         2009|
|     Margaret King|       Médio|    Chile|         2005|
|  Matthew Trembley|       Médio|  Uruguai|         2001|
|         Mary Cole|       Médio|  Uruguai|         2010|
|     Marita Hively|       Médio|  Bolívia|         2009|
|   Carissa Stanley|       Médio|    Chile|         2004|
|   Richard Bundage| Fundamental|  Uruguai|         2010|
|      Michael Dunn| Fundamental|  Uruguai|         2003|
|Marybeth Eberheart|       Médio|  Uruguai|         2007|
|   Deborah Lizotte|       Médio|Argentina|         2004|
+------------------+------------+---------+-------------+
only showing top 10 rows


In [9]:
df_millennials = df_nomes.filter(
    (F.col("AnoNascimento") >= 1980) & (F.col("AnoNascimento") <= 1994)
)

print("Número de Millennials:", df_millennials.count())

Número de Millennials: 2270443


In [12]:
df_millennials_sql = spark.sql("""
    SELECT COUNT(*) AS Qntd_millennials
    FROM pessoas
    WHERE AnoNascimento BETWEEN 1980 AND 1994
""")

df_millennials_sql.show()

+----------------+
|Qntd_millennials|
+----------------+
|         2270443|
+----------------+



In [16]:
df_geracoes = spark.sql("""
    SELECT
        Pais,
        CASE
            WHEN AnoNascimento BETWEEN 1944 AND 1964 THEN 'Baby Boomers'
            WHEN AnoNascimento BETWEEN 1965 AND 1979 THEN 'Geração X'
            WHEN AnoNascimento BETWEEN 1980 AND 1994 THEN 'Millennials'
            WHEN AnoNascimento BETWEEN 1995 AND 2015 THEN 'Geração Z'
            ELSE 'Outra'
        END AS Geracao,
        COUNT(*) AS Quantidade
    FROM pessoas
    GROUP BY Pais, Geracao
    ORDER BY Pais ASC, Geracao ASC, Quantidade ASC
""")

df_geracoes.show(52)

+---------------+------------+----------+
|           Pais|     Geracao|Quantidade|
+---------------+------------+----------+
|      Argentina|Baby Boomers|    430651|
|      Argentina|   Geração X|    322492|
|      Argentina|   Geração Z|    344238|
|      Argentina| Millennials|    321955|
|        Bolívia|Baby Boomers|    357844|
|        Bolívia|   Geração X|    269111|
|        Bolívia|   Geração Z|    286902|
|        Bolívia| Millennials|    268370|
|         Brasil|Baby Boomers|    232492|
|         Brasil|   Geração X|    174886|
|         Brasil|   Geração Z|    186090|
|         Brasil| Millennials|    173963|
|          Chile|Baby Boomers|    546459|
|          Chile|   Geração X|    408897|
|          Chile|   Geração Z|    437370|
|          Chile| Millennials|    409772|
|       Colômbia|Baby Boomers|     51813|
|       Colômbia|   Geração X|     38660|
|       Colômbia|   Geração Z|     41261|
|       Colômbia| Millennials|     38905|
|        Equador|Baby Boomers|    